# KBStats High Average Points Across the Last Five Match Slots

This notebook selects the newest timestamped KBStats player snapshot and retains players whose average across the five stored `history` match slots is at or above a configurable cutoff.

Every slot contributes to the denominator. A slot with `hasPlayed: false` or null points contributes zero points, so the calculation is always `sum(five slot scores) / 5`. With the default configuration, the cutoff is **90.75**. Results are displayed and exported to timestamped JSON and CSV files.

## 1. Resolve project paths

In [1]:
# Import the libraries required by this notebook step.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    DERIVED_KBSTATS_LAST_5_HIGH_AVERAGE_PLAYERS_DIR,
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)

## 2. Imports and configuration

Change `LAST_5_AVERAGE_POINTS_CUTOFF` to reuse the notebook with another threshold. `MATCH_SLOT_COUNT` remains five for this analysis.

In [2]:
# Import the libraries required by this notebook step.
from __future__ import annotations

import json
import math
import re
import warnings
from datetime import datetime, timezone
from typing import Any

import pandas as pd
from IPython.display import display

# Set workflow configuration value: LAST_5_AVERAGE_POINTS_CUTOFF.
LAST_5_AVERAGE_POINTS_CUTOFF = 90.75
# Set workflow configuration value: MATCH_SLOT_COUNT.
MATCH_SLOT_COUNT = 5

# Set workflow configuration value: KBSTATS_FILENAME_RE.
KBSTATS_FILENAME_RE = re.compile(
    r"^kbstats_players_"
    r"(?P<date>\d{8})_"
    r"(?P<time>\d{6})_"
    r"(?P<offset>[+-]\d{4})\.json$"
)

# Validate the input before continuing with later processing.
if (
    not isinstance(LAST_5_AVERAGE_POINTS_CUTOFF, (int, float))
    or isinstance(LAST_5_AVERAGE_POINTS_CUTOFF, bool)
    or not math.isfinite(float(LAST_5_AVERAGE_POINTS_CUTOFF))
):
    raise ValueError("LAST_5_AVERAGE_POINTS_CUTOFF must be a finite number.")
# Validate the input before continuing with later processing.
if MATCH_SLOT_COUNT != 5:
    raise ValueError("MATCH_SLOT_COUNT must remain 5 for this notebook.")

## 3. Select the newest KBStats snapshot

The newest input is determined from the timezone-aware filename timestamp, not modification time. Malformed candidates are reported and ignored; an ambiguous newest instant is rejected.

In [3]:
# Parse and validate kbstats filename timestamp for reuse in the workflow.
def parse_kbstats_filename_timestamp(path: Path) -> datetime:
    match = KBSTATS_FILENAME_RE.fullmatch(path.name)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(
            f"Filename does not contain a supported KBStats timestamp: {path.name}"
        )
    # Handle expected failures with a clear, actionable message.
    try:
        parsed = datetime.strptime(
            f"{match.group('date')}_{match.group('time')}_{match.group('offset')}",
            "%Y%m%d_%H%M%S_%z",
        )
    except ValueError as exc:
        raise ValueError(f"Invalid timestamp in {path.name}: {exc}") from exc
    return parsed.astimezone(timezone.utc)


# Select latest kbstats file for reuse in the workflow.
def select_latest_kbstats_file(directory: Path) -> Path:
    # Validate the input before continuing with later processing.
    if not directory.is_dir():
        raise FileNotFoundError(
            f"KBStats player output directory not found: {directory}. "
            "Run the KBStats player extraction notebook first."
        )
    candidates = sorted(directory.glob("kbstats_players_*.json"))
    # Validate the input before continuing with later processing.
    if not candidates:
        raise FileNotFoundError(
            f"No kbstats_players_*.json files were found in {directory}."
        )

    parsed_candidates: list[tuple[datetime, Path]] = []
    # Process each available item while preserving the current workflow state.
    for path in candidates:
        # Handle expected failures with a clear, actionable message.
        try:
            parsed_candidates.append((parse_kbstats_filename_timestamp(path), path))
        except ValueError as exc:
            warnings.warn(f"Ignoring {path.name}: {exc}", stacklevel=2)
    # Validate the input before continuing with later processing.
    if not parsed_candidates:
        raise ValueError(
            "KBStats JSON files were found, but none had a valid timezone-aware "
            "timestamp in the filename."
        )

    latest_timestamp = max(timestamp for timestamp, _ in parsed_candidates)
    latest_paths = [
        path for timestamp, path in parsed_candidates if timestamp == latest_timestamp
    ]
    # Validate the input before continuing with later processing.
    if len(latest_paths) != 1:
        names = ", ".join(path.name for path in latest_paths)
        raise RuntimeError(
            "Multiple KBStats files encode the same latest instant: " + names
        )
    return latest_paths[0]


selected_input_path = select_latest_kbstats_file(KBSTATS_PLAYERS_DIR)
print(f"Selected input: {selected_input_path}")

Selected input: C:\kickbase project\outputs\kbstats\players\kbstats_players_20260823_004913_+0200.json


## 4. Derive five-slot averages and apply the cutoff

The API supplies five history slots in its current most-recent-first order. The first five slots are used. Non-appearances contribute zero; a played slot must contain finite numeric points.

In [4]:
# Check whether finite number for reuse in the workflow.
def is_finite_number(value: Any) -> bool:
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and math.isfinite(float(value))
    )


# Handle last five for reuse in the workflow.
def derive_last_five(player: dict[str, Any]) -> dict[str, Any]:
    history = player.get("history")
    # Validate the input before continuing with later processing.
    if not isinstance(history, list) or len(history) < MATCH_SLOT_COUNT:
        raise ValueError("history must contain at least five match slots")

    slot_points: list[float] = []
    played_slots = 0
    # Process each available item while preserving the current workflow state.
    for slot_index, slot in enumerate(history[:MATCH_SLOT_COUNT]):
        # Validate the input before continuing with later processing.
        if not isinstance(slot, dict):
            raise ValueError(f"history slot {slot_index} is not an object")
        # Validate the input before continuing with later processing.
        if slot.get("hasPlayed") is True:
            points = slot.get("points")
            # Validate the input before continuing with later processing.
            if not is_finite_number(points):
                raise ValueError(
                    f"played history slot {slot_index} has non-numeric points"
                )
            slot_points.append(float(points))
            played_slots += 1
        else:
            slot_points.append(0.0)

    return {
        "last5Points": slot_points,
        "last5PlayedSlots": played_slots,
        "last5AveragePoints": sum(slot_points) / MATCH_SLOT_COUNT,
    }


# Handle expected failures with a clear, actionable message.
try:
    raw_players = json.loads(selected_input_path.read_text(encoding="utf-8"))
except UnicodeDecodeError as exc:
    raise ValueError(f"Input is not valid UTF-8: {selected_input_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Invalid JSON at line {exc.lineno}, column {exc.colno}: {selected_input_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read {selected_input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_players, list):
    raise TypeError(
        f"Expected a JSON list of player records, got {type(raw_players).__name__}."
    )

derived_players: list[dict[str, Any]] = []
invalid_records: list[dict[str, Any]] = []
# Process each available item while preserving the current workflow state.
for source_index, player in enumerate(raw_players):
    if not isinstance(player, dict):
        invalid_records.append({"source_index": source_index, "reason": "record is not an object"})
        continue
    # Handle expected failures with a clear, actionable message.
    try:
        derived_fields = derive_last_five(player)
    except ValueError as exc:
        invalid_records.append(
            {"source_index": source_index, "player_id": player.get("id"), "reason": str(exc)}
        )
        continue
    derived_players.append({**player, **derived_fields})

# Validate the input before continuing with later processing.
if not derived_players:
    raise ValueError("No player records contain five usable match slots.")

selected_players = sorted(
    (
        player
        for player in derived_players
        if player["last5AveragePoints"] >= float(LAST_5_AVERAGE_POINTS_CUTOFF)
    ),
    key=lambda player: (
        -player["last5AveragePoints"],
        -float(player.get("averagePoints") or 0),
        str(player.get("name") or "").casefold(),
    ),
)

display_rows = [
    {
        "player_id": player.get("id"),
        "name": player.get("name"),
        "team_id": player.get("teamId"),
        "position": player.get("position"),
        "last_5_points": player["last5Points"],
        "played_slots": player["last5PlayedSlots"],
        "last_5_average_points": player["last5AveragePoints"],
        "season_average_points": player.get("averagePoints"),
        "market_value": player.get("marketValue"),
    }
    for player in selected_players
]
selected_display_df = pd.DataFrame(display_rows)

print("Processing summary")
print("------------------")
print(f"Source records: {len(raw_players):,}")
print(f"Records with five usable slots: {len(derived_players):,}")
print(f"Excluded invalid records: {len(invalid_records):,}")
print(f"Last-five average cutoff: {float(LAST_5_AVERAGE_POINTS_CUTOFF):g}")
print(f"Selected players: {len(selected_players):,}")
display(selected_display_df)

Processing summary
------------------
Source records: 469
Records with five usable slots: 469
Excluded invalid records: 0
Last-five average cutoff: 90.75
Selected players: 66


,player_id,name,team_id,position,last_5_points,played_slots,last_5_average_points,season_average_points,market_value
0,8329,Michael Olise,2,3,"[81.0, 334.0, 290.0, 166.0, 152.0]",5,204.6,225.0,64765581.0
1,1685,Joshua Kimmich,2,3,"[317.0, 238.0, 83.0, 0.0, 215.0]",4,170.6,186.0,59816839.0
2,1639,Nadiem Amiri,18,3,"[254.0, 98.0, 168.0, 93.0, 229.0]",5,168.4,134.0,33544900.0
3,4199,Aleix García,7,3,"[229.0, 121.0, 214.0, 93.0, 181.0]",5,167.6,142.0,38695115.0
4,7226,Harry Kane,2,4,"[415.0, 53.0, 45.0, 150.0, 163.0]",5,165.2,216.0,68779146.0
...,...,...,...,...,...,...,...,...,...
61,2300,Josip Stanišić,2,2,"[70.0, 172.0, 46.0, 72.0, 105.0]",5,93.0,128.0,22439334.0
62,493,Michael Gregoritsch,13,4,"[-15.0, 273.0, 132.0, 75.0, 0.0]",4,93.0,74.0,10372917.0
63,11752,Warmed Omari,6,2,"[86.0, 85.0, 151.0, 82.0, 57.0]",5,92.2,60.0,3892142.0
64,577,Daniel Batz,15,1,"[282.0, 0.0, 45.0, 37.0, 95.0]",4,91.8,111.0,2448053.0


## 5. Export JSON and CSV

Each selected source record is augmented with `last5Points`, `last5PlayedSlots`, and `last5AveragePoints`. Nested fields are serialized as JSON strings in the CSV.

In [5]:
generated_datetime = datetime.now().astimezone()
generated_at = generated_datetime.isoformat(timespec="seconds")
output_timestamp = generated_datetime.strftime("%Y%m%d_%H%M%S_%z")
output_directory = ensure_directory(
    DERIVED_KBSTATS_LAST_5_HIGH_AVERAGE_PLAYERS_DIR
)
json_output_path = output_directory / f"kbstats_last_5_high_average_players_{output_timestamp}.json"
csv_output_path = output_directory / f"kbstats_last_5_high_average_players_{output_timestamp}.csv"

output_document = {
    "generated_at": generated_at,
    "source_file": selected_input_path.name,
    "match_slot_count": MATCH_SLOT_COUNT,
    "non_appearance_points": 0,
    "last_5_average_points_cutoff": float(LAST_5_AVERAGE_POINTS_CUTOFF),
    "valid_player_count": len(derived_players),
    "excluded_invalid_player_count": len(invalid_records),
    "selected_player_count": len(selected_players),
    "players": selected_players,
}

# Handle expected failures with a clear, actionable message.
try:
    json_output_path.write_text(
        json.dumps(output_document, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
except OSError as exc:
    raise OSError(f"Could not write JSON output {json_output_path}: {exc}") from exc

csv_df = pd.json_normalize(selected_players, sep=".")
# Process each available item while preserving the current workflow state.
for column in csv_df.columns:
    csv_df[column] = csv_df[column].map(
        lambda value: (
            json.dumps(value, ensure_ascii=False)
            if isinstance(value, (list, dict))
            else value
        )
    )
# Handle expected failures with a clear, actionable message.
try:
    csv_df.to_csv(csv_output_path, index=False, encoding="utf-8-sig")
except OSError as exc:
    raise OSError(f"Could not write CSV output {csv_output_path}: {exc}") from exc

print(f"JSON output: {json_output_path}")
print(f"CSV output:  {csv_output_path}")

JSON output: C:\kickbase project\outputs\derived\kbstats_last_5_high_average_players\kbstats_last_5_high_average_players_20260823_005043_+0200.json
CSV output:  C:\kickbase project\outputs\derived\kbstats_last_5_high_average_players\kbstats_last_5_high_average_players_20260823_005043_+0200.csv
